# Independent State-Aware Physics Model

## Objective

Build a **new model from our own SCADA pipeline**, without using the teammate's yaw values or change-point dates.

The model has three independent stages:

1. **Detect temporal regimes from SCADA only** using a short-window apparent-optimum signal and a penalized piecewise-constant segmentation.
2. **Estimate one apparent power-optimal angle per detected regime** by pooling SCADA inside the regime.
3. **Calibrate regime yaw levels from the three labelled turbines** using strict turbine-level validation.

The final model is

\[
\hat y_k
=
L_{\mathrm{global}}
+
\beta\,r_k,
\]

where

\[
L_{\mathrm{global}} = C - \theta^\star
\]

is the original long-term physics level, while

\[
r_k
=
-\left(\theta_k-\bar\theta_{\mathrm{state}}\right)
\]

is the regime-specific physics residual derived from our own power-vs-vane estimator.

No teammate prediction values, cluster labels, or state boundaries are used anywhere in this notebook.


In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------
# Fixed local project paths
# ---------------------------------------------------------------------
ROOT = Path(r"E:/EnergyHacks/github_release")
SRC = ROOT / "src"
ZIP_PATH = Path(r"E:/EnergyHacks/turbines_data.zip")

if not ROOT.exists():
    raise FileNotFoundError(f"Project repo not found: {ROOT}")

if not SRC.exists():
    raise FileNotFoundError(f"Source directory not found: {SRC}")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Challenge data archive not found: {ZIP_PATH}"
    )

# Put repo source code first on the Python import path.
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT))

from baseline_loto_ridge import read_turbine
import baseline_openoa_power_vane_updated as b0mod

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
VALIDATE_TARGET = "PPP_WTG17"
FINAL_TARGET = "SSS_WTG06"

print("ROOT:", ROOT)
print("SRC:", SRC)
print("ZIP_PATH:", ZIP_PATH)


## 1. Fixed physics settings

We keep the original operating filter and empirical power-vs-vane estimator.

Two time scales are used:

- **7-day rolling window** for change-point detection: faster temporal response.
- **21-day rolling window** for the global long-term reference: consistent with the frozen B0 model.

The detected regimes are then re-estimated by pooling SCADA within each regime, which reduces daily argmax noise.


In [ ]:
B0_KWARGS = dict(
    wind_bin_width=1.0,
    angle_bin_width=1.0,
    min_samples_per_angle_bin=20,
    min_angle_bins=8,
    gamma_min=-30.0,
    gamma_max=30.0,
)

DETECT_WINDOW_DAYS = 7
GLOBAL_WINDOW_DAYS = 21

# Minimum calendar duration of a predicted regime.
MIN_SEG_DAYS = 14

# Trim regime edges before estimating the regime-level power optimum.
# This reduces contamination from the rolling-window transition zone.
STATE_EDGE_TRIM_DAYS = 5

# Segmentation penalty grid.  The apparent-optimum series is robustly
# standardized before segmentation, so the penalty scale is comparable.
PENALTY_GRID = [2, 4, 6, 8, 12, 16, 24, 32, 48, 64]

# Cap long-state weights so one very long plateau cannot dominate beta.
STATE_WEIGHT_CAP = 90


## 2. Build our own daily apparent-optimum signal

For every date \(t\), use the surrounding SCADA window and compute the B0 apparent power optimum:

\[
\theta_t
=
\arg\max_{\gamma\text{-bin}}
\operatorname{median}(P_{\mathrm{norm}}).
\]

This is an observed-bin argmax, not a fitted quadratic vertex.


In [ ]:
def prepare_turbine(turbine_id):
    raw = read_turbine(str(ZIP_PATH), turbine_id)
    data = b0mod.prepare(raw)
    return raw, data


def rolling_b0(data, window_days):
    half_before = (window_days - 1) // 2
    half_after = window_days - 1 - half_before

    unique_dates = (
        data[["date", "date_dt"]]
        .drop_duplicates()
        .sort_values("date_dt")
        .reset_index(drop=True)
    )

    rows = []
    for row in unique_dates.itertuples(index=False):
        center = pd.Timestamp(row.date_dt)
        start = center - pd.Timedelta(days=half_before)
        end = center + pd.Timedelta(days=half_after)

        window = data[
            data["date_dt"].between(start, end)
        ]

        result = b0mod.estimate_peak_angle(
            window,
            **B0_KWARGS,
        )

        rows.append({
            "date": center.normalize(),
            "theta": result["theta_hat_argmax"],
            "n_curve_bins": result["n_curve_bins"],
            "observability": result["argmax_observability"],
            "gamma_span": result["gamma_span"],
            "status": result["status"],
        })

    return pd.DataFrame(rows)


def daily_labels(raw):
    if "yaw_misalignment_deg" not in raw.columns:
        return None

    out = (
        raw.groupby("date", as_index=False)["yaw_misalignment_deg"]
           .median()
           .rename(columns={"yaw_misalignment_deg": "target"})
    )
    out["date"] = pd.to_datetime(out["date"])
    return out


raw_cache = {}
prepared_cache = {}
theta7_cache = {}
theta21_cache = {}
label_cache = {}

for turbine in TRAIN + [VALIDATE_TARGET, FINAL_TARGET]:
    raw, prepared = prepare_turbine(turbine)
    raw_cache[turbine] = raw
    prepared_cache[turbine] = prepared
    theta7_cache[turbine] = rolling_b0(prepared, DETECT_WINDOW_DAYS)
    theta21_cache[turbine] = rolling_b0(prepared, GLOBAL_WINDOW_DAYS)
    label_cache[turbine] = daily_labels(raw)

    print(
        turbine,
        "7d valid =", theta7_cache[turbine]["theta"].notna().sum(),
        "21d valid =", theta21_cache[turbine]["theta"].notna().sum(),
    )


## 3. Unsupervised regime detection

We fit a one-dimensional Potts / change-point model to the 7-day apparent-optimum series:

\[
\min_{z(t)}
\sum_t w_t\left(x_t-z_t\right)^2
+
\lambda\,N_{\mathrm{changes}}.
\]

Important properties:

- segmentation uses **SCADA only**
- no target labels are used to place change points
- no teammate change-point dates are used
- a minimum 14-day regime duration prevents daily jitter
- reliability weights downweight poorly observed B0 windows

The dynamic program below solves the penalized piecewise-constant problem exactly for a fixed penalty.


In [ ]:
def _robust_scale(values):
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - med))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale < 1e-6:
        scale = np.nanstd(values)
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    return med, scale


def prepare_segmentation_series(theta_frame):
    x = theta_frame.copy().sort_values("date")
    full_dates = pd.DataFrame({
        "date": pd.date_range(
            x["date"].min(),
            x["date"].max(),
            freq="D",
        )
    })
    x = full_dates.merge(x, on="date", how="left")

    # Interpolate the apparent optimum only for the segmentation objective.
    # Regime-level theta is later re-estimated directly from pooled SCADA.
    x["theta_filled"] = (
        x["theta"]
        .interpolate(limit_direction="both")
    )

    med, scale = _robust_scale(x["theta_filled"])
    x["z"] = (x["theta_filled"] - med) / scale

    # Simple B0 reliability weight.
    bins = x["n_curve_bins"].fillna(0).clip(lower=0)
    obs = x["observability"].fillna(0).clip(lower=0)

    w_bins = (bins / 20.0).clip(upper=1.0)
    w_obs = (obs / 8.0).clip(upper=1.0)

    x["weight"] = (0.25 + 0.75 * w_bins * w_obs).astype(float)

    return x, med, scale


def potts_segments(theta_frame, penalty, min_seg_days=14):
    x, med, scale = prepare_segmentation_series(theta_frame)

    z = x["z"].to_numpy(dtype=float)
    w = x["weight"].to_numpy(dtype=float)
    n = len(x)

    # Prefix sums give weighted SSE for any [i, j) interval in O(1).
    pw = np.r_[0.0, np.cumsum(w)]
    pwz = np.r_[0.0, np.cumsum(w * z)]
    pwz2 = np.r_[0.0, np.cumsum(w * z * z)]

    def seg_cost(i, j):
        sw = pw[j] - pw[i]
        if sw <= 1e-12:
            return 0.0
        swz = pwz[j] - pwz[i]
        swz2 = pwz2[j] - pwz2[i]
        return float(swz2 - swz * swz / sw)

    inf = float("inf")
    dp = np.full(n + 1, inf)
    prev = np.full(n + 1, -1, dtype=int)

    # First segment is not charged a change penalty.
    dp[0] = -float(penalty)

    for end in range(min_seg_days, n + 1):
        start_max = end - min_seg_days
        best_cost = inf
        best_start = -1

        for start in range(0, start_max + 1):
            if start != 0 and start < min_seg_days:
                continue
            if not np.isfinite(dp[start]):
                continue

            value = (
                dp[start]
                + seg_cost(start, end)
                + float(penalty)
            )

            if value < best_cost:
                best_cost = value
                best_start = start

        dp[end] = best_cost
        prev[end] = best_start

    # Backtrack boundaries.
    spans = []
    end = n
    if prev[end] < 0:
        raise RuntimeError("Segmentation failed")

    while end > 0:
        start = int(prev[end])
        spans.append((start, end))
        end = start

    spans.reverse()

    rows = []
    for cluster, (start, end) in enumerate(spans):
        part = x.iloc[start:end]
        rows.append({
            "cluster": cluster,
            "start": part["date"].iloc[0],
            "end": part["date"].iloc[-1],
            "days": len(part),
            "theta7_median": float(part["theta"].median()),
        })

    return pd.DataFrame(rows), x


## 4. Re-estimate one physics optimum per detected regime

After segmentation, do **not** use the segmented 7-day value as the yaw level.

Instead:

1. trim a few days from each state edge;
2. pool the operating SCADA rows inside the state;
3. run the original empirical power-vs-vane estimator once on the pooled state.

This gives a much more stable state-level apparent optimum \(\theta_k\).


In [ ]:
def estimate_state_theta(turbine_id, segments):
    data = prepared_cache[turbine_id]
    labels = label_cache[turbine_id]

    rows = []

    for seg in segments.itertuples(index=False):
        start = pd.Timestamp(seg.start)
        end = pd.Timestamp(seg.end)
        days = int(seg.days)

        trim = min(
            STATE_EDGE_TRIM_DAYS,
            max(0, (days - 7) // 4),
        )

        inner_start = start + pd.Timedelta(days=trim)
        inner_end = end - pd.Timedelta(days=trim)

        window = data[
            data["date_dt"].between(inner_start, inner_end)
        ]

        result = b0mod.estimate_peak_angle(
            window,
            **B0_KWARGS,
        )

        theta_state = result["theta_hat_argmax"]

        # Fallback: median 21d B0 inside the state if pooled state is
        # unexpectedly unobservable.
        if not np.isfinite(theta_state):
            q = theta21_cache[turbine_id]
            theta_state = float(
                q[q["date"].between(start, end)]["theta"].median()
            )

        row = {
            "cluster": int(seg.cluster),
            "start": start,
            "end": end,
            "days": days,
            "theta_state": float(theta_state),
            "n_curve_bins": int(result["n_curve_bins"]),
            "observability": result["argmax_observability"],
        }

        if labels is not None:
            y = labels[
                labels["date"].between(start, end)
            ]["target"].dropna()

            row["target_state"] = (
                float(y.median()) if len(y) else np.nan
            )
            row["n_label_days"] = int(len(y))

        rows.append(row)

    states = pd.DataFrame(rows)

    # Duration-weighted state center.  Subtracting this makes the temporal
    # residual exactly zero-mean over the two-year calendar.
    w = states["days"].to_numpy(dtype=float)
    theta_center = float(
        np.average(states["theta_state"], weights=w)
    )

    states["physics_residual"] = -(
        states["theta_state"] - theta_center
    )

    return states


def global_theta_star(turbine_id):
    return float(
        theta21_cache[turbine_id]["theta"]
        .dropna()
        .median()
    )


def target_global_mean(turbine_id):
    labels = label_cache[turbine_id]
    if labels is None:
        return np.nan
    return float(labels["target"].dropna().mean())


## 5. Calibrate \(C\) and state amplitude \(\beta\)

The global level keeps the original form:

\[
L = C-\theta^\star.
\]

For training turbine \(j\):

\[
C_j = \bar y_j+\theta^\star_j.
\]

The temporal state correction is:

\[
\Delta y_k
=
\beta\,r_k.
\]

We fit \(\beta\) through zero on labelled states. This preserves the separation between:

- **global offset calibration** \(C\)
- **within-turbine state amplitude** \(\beta\)

and avoids fitting a second arbitrary intercept.


In [ ]:
segment_cache = {}
state_cache = {}

def get_segments(turbine_id, penalty):
    key = (turbine_id, float(penalty))
    if key not in segment_cache:
        segment_cache[key] = potts_segments(
            theta7_cache[turbine_id],
            penalty=float(penalty),
            min_seg_days=MIN_SEG_DAYS,
        )[0]
    return segment_cache[key].copy()


def get_states(turbine_id, penalty):
    key = (turbine_id, float(penalty))
    if key not in state_cache:
        state_cache[key] = estimate_state_theta(
            turbine_id,
            get_segments(turbine_id, penalty),
        )
    return state_cache[key].copy()


def fit_C(train_ids):
    values = [
        target_global_mean(t)
        + global_theta_star(t)
        for t in train_ids
    ]
    return float(np.mean(values))


def fit_beta(train_ids, penalty):
    rows = []

    for t in train_ids:
        states = get_states(t, penalty)
        y0 = target_global_mean(t)

        tmp = states.dropna(
            subset=["target_state", "physics_residual"]
        ).copy()
        tmp["target_residual"] = tmp["target_state"] - y0
        tmp["weight"] = np.minimum(
            tmp["days"].to_numpy(dtype=float),
            STATE_WEIGHT_CAP,
        )
        rows.append(tmp)

    fit = pd.concat(rows, ignore_index=True)

    x = fit["physics_residual"].to_numpy(dtype=float)
    y = fit["target_residual"].to_numpy(dtype=float)
    w = fit["weight"].to_numpy(dtype=float)

    denom = np.sum(w * x * x)
    if denom <= 1e-12:
        return 0.0

    beta = float(np.sum(w * x * y) / denom)

    # Conservative physical bound.
    return float(np.clip(beta, 0.0, 2.0))


def expand_predictions(turbine_id, states, global_level, beta):
    dates = pd.DataFrame({
        "date": pd.date_range(
            theta7_cache[turbine_id]["date"].min(),
            theta7_cache[turbine_id]["date"].max(),
            freq="D",
        )
    })

    dates["prediction"] = np.nan
    dates["cluster"] = pd.Series(
        pd.NA, index=dates.index, dtype="Int64"
    )

    for row in states.itertuples(index=False):
        mask = dates["date"].between(row.start, row.end)
        dates.loc[mask, "prediction"] = (
            global_level
            + beta * float(row.physics_residual)
        )
        dates.loc[mask, "cluster"] = int(row.cluster)

    assert dates["prediction"].notna().all()
    assert dates["cluster"].notna().all()

    return dates


def score_daily(turbine_id, predictions):
    labels = label_cache[turbine_id].dropna(
        subset=["target"]
    )

    scored = labels.merge(
        predictions[["date", "prediction"]],
        on="date",
        how="inner",
        validate="one_to_one",
    )

    err = (
        scored["prediction"].to_numpy(dtype=float)
        - scored["target"].to_numpy(dtype=float)
    )

    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err**2))),
        "bias": float(np.mean(err)),
        "n": int(len(err)),
    }


## 6. Strict outer LOTO

This is the main gate.

For each held-out turbine:

1. choose the segmentation penalty using only the other two labelled turbines;
2. fit \(C\) and \(\beta\) using only those two turbines;
3. segment the held-out turbine from its SCADA only;
4. predict its complete daily yaw trajectory;
5. score against its labels.

For penalty selection inside each outer fold, the two available training turbines are cross-predicted against each other. This is small-sample nested validation, but it prevents the outer holdout labels from influencing the chosen penalty.


In [ ]:
def inner_penalty_score(train_ids, penalty):
    maes = []

    # With two outer-training turbines, use each one once as an inner holdout.
    for inner_holdout in train_ids:
        inner_train = [t for t in train_ids if t != inner_holdout]

        C = fit_C(inner_train)
        beta = fit_beta(inner_train, penalty)

        L = C - global_theta_star(inner_holdout)
        states = get_states(inner_holdout, penalty)
        pred = expand_predictions(
            inner_holdout,
            states,
            global_level=L,
            beta=beta,
        )

        maes.append(
            score_daily(inner_holdout, pred)["mae"]
        )

    return float(np.mean(maes))


outer_rows = []

for holdout in TRAIN:
    train_ids = [t for t in TRAIN if t != holdout]

    penalty_scores = {
        penalty: inner_penalty_score(train_ids, penalty)
        for penalty in PENALTY_GRID
    }

    best_penalty = min(
        penalty_scores,
        key=penalty_scores.get,
    )

    C = fit_C(train_ids)
    beta = fit_beta(train_ids, best_penalty)

    L = C - global_theta_star(holdout)
    states = get_states(holdout, best_penalty)

    pred_state = expand_predictions(
        holdout,
        states,
        global_level=L,
        beta=beta,
    )
    m_state = score_daily(holdout, pred_state)

    # Same outer-fold global calibration, but no temporal state correction.
    pred_constant = pred_state.copy()
    pred_constant["prediction"] = L
    m_const = score_daily(holdout, pred_constant)

    # Diagnostic: raw physics state amplitude beta=1.
    pred_beta1 = expand_predictions(
        holdout,
        states,
        global_level=L,
        beta=1.0,
    )
    m_beta1 = score_daily(holdout, pred_beta1)

    outer_rows.append({
        "holdout": holdout,
        "penalty": best_penalty,
        "beta": beta,
        "n_segments": len(states),
        "constant_mae": m_const["mae"],
        "beta1_mae": m_beta1["mae"],
        "state_mae": m_state["mae"],
        "constant_rmse": m_const["rmse"],
        "beta1_rmse": m_beta1["rmse"],
        "state_rmse": m_state["rmse"],
        "state_bias": m_state["bias"],
    })

outer = pd.DataFrame(outer_rows)
display(outer.round(3))

print("Nested outer macro MAE")
print("  constant:", outer["constant_mae"].mean())
print("  beta=1  :", outer["beta1_mae"].mean())
print("  fitted  :", outer["state_mae"].mean())

print("\nNested outer macro RMSE")
print("  constant:", outer["constant_rmse"].mean())
print("  beta=1  :", outer["beta1_rmse"].mean())
print("  fitted  :", outer["state_rmse"].mean())


### Model acceptance rule

Do **not** submit this model merely because one fold improves.

A useful independent state model should show:

- state-aware MAE better than the constant baseline in most or all outer folds;
- fitted \(\beta\) values that remain physically plausible and not wildly unstable;
- a reasonable number of long-lived regimes rather than dozens of short segments.

If strict LOTO fails this gate, stop here and do not use the public leaderboard to rescue the model.


## 7. Select the final segmentation penalty from labelled turbines only

For deployment we need one fixed penalty. We select it by ordinary three-turbine LOTO on the labelled set only.

This selection is separate from the nested outer score above:

- **nested outer result** = honest model-development diagnostic;
- **three-turbine CV penalty** = final training choice before target deployment.


In [ ]:
penalty_rows = []

for penalty in PENALTY_GRID:
    fold_maes = []

    for holdout in TRAIN:
        train_ids = [t for t in TRAIN if t != holdout]

        C = fit_C(train_ids)
        beta = fit_beta(train_ids, penalty)
        L = C - global_theta_star(holdout)

        pred = expand_predictions(
            holdout,
            get_states(holdout, penalty),
            global_level=L,
            beta=beta,
        )

        fold_maes.append(
            score_daily(holdout, pred)["mae"]
        )

    penalty_rows.append({
        "penalty": penalty,
        "macro_mae": float(np.mean(fold_maes)),
        "fold_maes": fold_maes,
    })

penalty_table = pd.DataFrame(penalty_rows)
display(penalty_table[["penalty", "macro_mae"]].round(3))

FINAL_PENALTY = float(
    penalty_table.loc[
        penalty_table["macro_mae"].idxmin(),
        "penalty",
    ]
)

FINAL_C = fit_C(TRAIN)
FINAL_BETA = fit_beta(TRAIN, FINAL_PENALTY)

print("FINAL_PENALTY:", FINAL_PENALTY)
print("FINAL_C:", FINAL_C)
print("FINAL_BETA:", FINAL_BETA)


## 8. Deploy the independent model to PPP_WTG17 and SSS_WTG06

No target labels and no teammate outputs are used.

For each target:

1. segment its 7-day B0 apparent-optimum series;
2. estimate one pooled SCADA optimum per segment;
3. compute the original 21-day long-term \(\theta^\star\);
4. place the state residuals around the independently calibrated global level.


In [ ]:
def deploy_target(turbine_id):
    segments = get_segments(
        turbine_id,
        FINAL_PENALTY,
    )
    states = estimate_state_theta(
        turbine_id,
        segments,
    )

    L = FINAL_C - global_theta_star(turbine_id)

    pred = expand_predictions(
        turbine_id,
        states,
        global_level=L,
        beta=FINAL_BETA,
    )

    return {
        "segments": segments,
        "states": states,
        "global_level": L,
        "prediction": pred,
    }


validate_model = deploy_target(VALIDATE_TARGET)
final_model = deploy_target(FINAL_TARGET)

print("PPP_WTG17 global level:",
      validate_model["global_level"])
print("PPP_WTG17 segments:",
      len(validate_model["states"]))
display(validate_model["states"].round(3))

print("\nSSS_WTG06 global level:",
      final_model["global_level"])
print("SSS_WTG06 segments:",
      len(final_model["states"]))
display(final_model["states"].round(3))


## 9. Generate isolated candidate files

These are **new independent-model candidates**.

- validation candidate: `Results_33_T3_2.csv`
- final research candidate: `SSS_WTG06_independent_state_physics_candidate.csv`

The final candidate deliberately uses a non-organizer filename until the replacement-final naming procedure is confirmed.


In [ ]:
def template_locked_submission(
    prediction,
    template_path,
    turbine_id,
):
    template = pd.read_csv(template_path)
    template["date"] = pd.to_datetime(template["date"])

    pred = prediction.copy()
    pred["date"] = pd.to_datetime(pred["date"])

    out = template[["turbine_id", "date"]].merge(
        pred.rename(columns={
            "prediction": "yaw_misalignment_deg"
        })[["date", "yaw_misalignment_deg", "cluster"]],
        on="date",
        how="left",
        validate="one_to_one",
    )

    assert len(out) == 731
    assert out["turbine_id"].eq(turbine_id).all()
    assert out["yaw_misalignment_deg"].notna().all()
    assert np.isfinite(out["yaw_misalignment_deg"]).all()
    assert out["yaw_misalignment_deg"].abs().max() <= 90
    assert out["cluster"].notna().all()

    out["date"] = out["date"].dt.strftime("%Y-%m-%d")
    out["cluster"] = out["cluster"].astype(int)

    return out


validate_template = next(
    p for p in [
        ROOT / "submission_template_validate.csv",
        ROOT / "data" / "submission_template_validate.csv",
    ]
    if p.exists()
)

final_template = next(
    p for p in [
        ROOT / "submission_template_final.csv",
        ROOT / "data" / "submission_template_final.csv",
    ]
    if p.exists()
)

validate_out = template_locked_submission(
    validate_model["prediction"],
    validate_template,
    VALIDATE_TARGET,
)

final_out = template_locked_submission(
    final_model["prediction"],
    final_template,
    FINAL_TARGET,
)

validate_path = ROOT / "Results_33_T3_2.csv"
final_path = ROOT / "SSS_WTG06_independent_state_physics_candidate.csv"

validate_out.to_csv(
    validate_path,
    index=False,
    float_format="%.6f",
)
final_out.to_csv(
    final_path,
    index=False,
    float_format="%.6f",
)

print("Wrote:", validate_path)
print("Wrote:", final_path)

print(
    "Validation:",
    validate_out["cluster"].nunique(),
    "clusters,",
    validate_out["yaw_misalignment_deg"].nunique(),
    "yaw levels",
)
print(
    "Final:",
    final_out["cluster"].nunique(),
    "clusters,",
    final_out["yaw_misalignment_deg"].nunique(),
    "yaw levels",
)


## 10. What to inspect before any submission

The three most important outputs are:

1. `outer` — strict nested turbine-level validation;
2. `FINAL_PENALTY` and `FINAL_BETA` — whether the learned temporal model is stable;
3. `validate_model["states"]` — whether the independent PPP_WTG17 segmentation is physically plausible.

Only if the strict labelled-turbine evidence is convincing should `Results_33_T3_2.csv` be considered for the public leaderboard.

This notebook intentionally avoids using the already-known public leaderboard metrics to solve for hidden yaw levels.
